# Demonstration quality check

Loads the raw recordings in `A2/data/raw/<name>/` (JPEGs + `data.npz`) and checks:

1. **Timing**: frame rate, dropped frames, image/joint counts
2. **Joints**: trajectories, leader–follower tracking, start/end poses, spikes
3. **Gripper**: grasp and release detected in every demonstration
4. **Images**: scene diversity, frame strips, brightness, frozen frames

`action` = **leader** joint positions (what ACT learns to output); `state` = **follower** joint positions (what ACT sees as proprioception).

In [ ]:
import json, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2

ROOT = Path("../data/raw/so101_pick_place")
meta = json.loads((ROOT / "meta.json").read_text())
FPS, JOINTS = meta["fps"], meta["joints"]
log = pd.read_csv(ROOT / "demos.csv")

episodes = sorted(ROOT.glob("episode_*"))
EP = {}
for p in episodes:
    d = np.load(p / "data.npz")
    EP[int(p.name.split("_")[1])] = dict(path=p, state=d["state"], action=d["action"], t=d["timestamp"])
print(f"{len(EP)} episodes, {sum(len(e['t']) for e in EP.values())} frames, "
      f"{sum(len(e['t']) for e in EP.values()) / FPS / 60:.1f} min")
print("log status:", log.status.value_counts().to_dict())

## 1. Timing and file integrity

In [ ]:
def idle_frames(a, thr=0.3):
    """Frames at the start / end where the leader arm (gripper excluded) is not moving."""
    v = np.abs(np.diff(a[:, :5], axis=0)).max(1)
    moving = np.where(v > thr)[0]
    return int(moving[0]), int(len(v) - moving[-1])

def runs(mask, gap=5):
    """(start, end) of runs of True, bridging gaps of up to `gap` frames."""
    out, start, last = [], None, None
    for i in np.where(mask)[0]:
        if start is None or i - last > gap:
            if start is not None: out.append((start, last))
            start = i
        last = i
    return out + ([(start, last)] if start is not None else [])

def holds(s, a):
    """Periods where the follower gripper is stopped by the cube: it sits at the cube's width
    (7-17) and does not move, while the leader is at or below it. Returns runs of >= 1/3 s."""
    gs, ga = s[:, 5], a[:, 5]
    held = (gs > 7) & (gs < 17) & (ga <= gs + 0.5) & np.r_[True, np.abs(np.diff(gs)) < 1]
    return [r for r in runs(held) if r[1] - r[0] >= FPS // 3]

def grasp_release(s, a):
    """(grasp frame, release frame) of the longest hold, or (None, None)."""
    h = [r for r in holds(s, a) if r[1] - r[0] >= FPS]
    return max(h, key=lambda r: r[1] - r[0]) if h else (None, None)

rows = []
for i, e in EP.items():
    t, s, a = e["t"], e["state"], e["action"]
    dt = np.diff(t) * 1000
    n_w = len(list((e["path"] / "wrist").glob("*.jpg")))
    n_s = len(list((e["path"] / "scene").glob("*.jpg")))
    i0, i1 = idle_frames(a)
    g, r = grasp_release(s, a)
    rows.append(dict(ep=i, frames=len(t), dur_s=round(t[-1] - t[0], 1),
                     dt_ms=round(dt.mean(), 1), dt_max=round(dt.max(), 1), late=(dt > 1.5e3 / FPS).sum(),
                     imgs_ok=(n_w == n_s == len(t)), idle_start_s=round(i0 / FPS, 1), idle_end_s=round(i1 / FPS, 1),
                     grasp_s=None if g is None else round(g / FPS, 1), release_s=None if r is None else round(r / FPS, 1),
                     n_holds=len(holds(s, a)), grip_squeeze=None if g is None else round(float(np.median(s[g:r, 5] - a[g:r, 5])), 1),
                     grip_start=round(a[0, 5]), grip_end=round(a[-1, 5])))
summary = pd.DataFrame(rows).set_index("ep")
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].hist(np.concatenate([np.diff(e["t"]) * 1000 for e in EP.values()]), bins=50)
ax[0].axvline(1000 / FPS, c="r", ls="--"); ax[0].set(xlabel="frame interval (ms)", ylabel="count", title="Frame timing")
ax[1].bar(summary.index, summary.dur_s); ax[1].set(xlabel="episode", ylabel="s", title="Episode duration")
plt.tight_layout()

## 2. Joint trajectories
All episodes overlaid, time normalised to 0–1. Solid = leader action, the policy's target.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharex=True)
for i, e in EP.items():
    x = np.linspace(0, 1, len(e["t"]))
    for j, ax in enumerate(axes.flat):
        ax.plot(x, e["action"][:, j], lw=0.8, alpha=0.6)
for j, ax in enumerate(axes.flat):
    ax.set_title(JOINTS[j]); ax.set_xlabel("normalised time")
axes[0, 0].set_ylabel("deg"); axes[1, 0].set_ylabel("deg (gripper 0–100)")
plt.tight_layout()

### One episode in detail: leader (action) vs follower (state)
Change `EPISODE` to look at another one. Grey lines mark the detected grasp and release.

In [ ]:
EPISODE = 5
e = EP[EPISODE]; t = e["t"] - e["t"][0]
g, r = grasp_release(e["state"], e["action"])
fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharex=True)
for j, ax in enumerate(axes.flat):
    ax.plot(t, e["action"][:, j], label="leader (action)")
    ax.plot(t, e["state"][:, j], label="follower (state)", alpha=0.8)
    for f in (g, r):
        if f is not None: ax.axvline(t[f], c="grey", ls="--")
    ax.set_title(JOINTS[j])
axes[0, 0].legend(); [ax.set_xlabel("s") for ax in axes[1]]
plt.tight_layout()

### Leader–follower tracking
The follower lags the leader by a few frames and sags a little under gravity. This is normal: ACT is trained to output the **leader** command, and at test time the follower lags it in the same way.

In [ ]:
lag_rows = []
for i, e in EP.items():
    s, a = e["state"], e["action"]
    row = {"ep": i}
    for j in range(5):
        errs = [np.abs(a[: len(a) - k, j] - s[k:, j]).mean() for k in range(12)]
        row[f"{JOINTS[j]}_lag_fr"] = int(np.argmin(errs))
        row[f"{JOINTS[j]}_offset"] = round(float(np.mean(a[: len(a) - np.argmin(errs), j] - s[np.argmin(errs):, j])), 1)
    lag_rows.append(row)
lag = pd.DataFrame(lag_rows).set_index("ep")
print("median lag (frames):", lag.filter(like="lag").median().to_dict())
print("median steady offset leader-follower (deg):", lag.filter(like="offset").median().to_dict())

### Start and end poses
Consistent start poses make the policy's first moves predictable.

In [ ]:
start = np.stack([e["state"][0] for e in EP.values()]); end = np.stack([e["state"][-1] for e in EP.values()])
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for j, ax in enumerate(axes):
    ax.scatter(np.zeros(len(start)), start[:, j], label="start"); ax.scatter(np.ones(len(end)), end[:, j], label="end")
    ax.set_xticks([0, 1], ["start", "end"]); ax.set_title(JOINTS[j]); ax.set_xlim(-0.5, 1.5)
plt.tight_layout()
print(pd.DataFrame({"start_std": start.std(0), "end_std": end.std(0)}, index=JOINTS).round(1))

### Spikes
Largest per-frame jump in the leader action. Jumps of more than ~10° in one frame usually mean a glitch, not a real motion.

In [ ]:
jump = pd.DataFrame({i: np.abs(np.diff(e["action"], axis=0)).max(0) for i, e in EP.items()}, index=JOINTS).T.round(1)
jump.style.background_gradient(axis=None, cmap="Reds")

## 3. Gripper: grasp, hold and release
While the cube is held, the follower gripper stops at the cube's width (~10–14) while the leader is squeezed further closed. The gap between them (`grip_squeeze` in the summary table) is how hard the cube is gripped: the motor's force grows with it. A squeeze near 0 means the cube is barely held.

In [ ]:
n = len(EP); cols = 7
fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(18, 2.2 * np.ceil(n / cols)), sharey=True)
for ax, (i, e) in zip(axes.flat, EP.items()):
    t = e["t"] - e["t"][0]
    ax.plot(t, e["action"][:, 5], label="leader"); ax.plot(t, e["state"][:, 5], label="follower")
    g, r = grasp_release(e["state"], e["action"])
    if g is not None: ax.axvspan(t[g], t[r], color="green", alpha=0.15)
    ax.set_title(f"ep {i}" + ("" if g is not None else "  NO GRASP"), fontsize=9)
for ax in axes.flat[n:]: ax.axis("off")
axes.flat[0].legend(fontsize=7); plt.tight_layout()

In [ ]:
# After the release the arm should lift (shoulder_lift / elbow change) before the episode ends.
post = []
for i, e in EP.items():
    g, r = grasp_release(e["state"], e["action"])
    if r is None: post.append(dict(ep=i)); continue
    a = e["action"][r:]
    post.append(dict(ep=i, s_after_release=round(len(a) / FPS, 1),
                     **{f"{j}_moved": round(float(np.abs(a[:, k] - a[0, k]).max())) for k, j in enumerate(JOINTS[:3])}))
pd.DataFrame(post).set_index("ep").T

## 4. Images

### Scene diversity: the first scene frame of each episode

In [ ]:
def load(ep, cam, idx, size=(320, 240)):
    im = cv2.imread(str(EP[ep]["path"] / cam / f"{idx:06d}.jpg"))
    return cv2.cvtColor(cv2.resize(im, size), cv2.COLOR_BGR2RGB)

cols = 6
fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(18, 2.4 * np.ceil(n / cols)))
for ax, i in zip(axes.flat, EP):
    ax.imshow(load(i, "scene", 0)); ax.set_title(f"ep {i}", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout()

### Frame strip for one episode
Wrist (top) and scene (bottom) at evenly spaced times.

In [ ]:
EPISODE = 5; K = 8
N = len(EP[EPISODE]["t"]); idx = np.linspace(0, N - 1, K).astype(int)
fig, axes = plt.subplots(2, K, figsize=(2.3 * K, 4))
for k, f in enumerate(idx):
    axes[0, k].imshow(load(EPISODE, "wrist", f)); axes[1, k].imshow(load(EPISODE, "scene", f))
    axes[0, k].set_title(f"{f / FPS:.1f}s", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout()

### Brightness and frozen frames
A camera that stops delivering new frames shows up as long runs of identical images while the arm is moving.

In [ ]:
def image_stats(ep, cam, step=1):
    prev, bright, same = None, [], []
    for f in range(0, len(EP[ep]["t"]), step):
        im = cv2.resize(cv2.imread(str(EP[ep]["path"] / cam / f"{f:06d}.jpg"), cv2.IMREAD_GRAYSCALE), (160, 120)).astype(np.float32)
        bright.append(im.mean()); same.append(prev is not None and np.abs(im - prev).mean() < 0.2); prev = im
    return np.array(bright), np.array(same)

def longest_run(x):
    best = run = 0
    for v in x: run = run + 1 if v else 0; best = max(best, run)
    return best

img_rows, bright = [], {}
for i in EP:
    row = {"ep": i}
    for cam in meta["cameras"]:
        b, same = image_stats(i, cam); bright[(i, cam)] = b
        moving = np.r_[False, np.abs(np.diff(EP[i]["action"][:, :5], axis=0)).max(1) > 0.5]
        row[f"{cam}_frozen_while_moving"] = int((same & moving).sum())
        row[f"{cam}_longest_frozen_run"] = longest_run(same & moving)
        row[f"{cam}_brightness"] = f"{b.min():.0f}-{b.max():.0f}"
    img_rows.append(row)
pd.DataFrame(img_rows).set_index("ep")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3), sharey=True)
for k, cam in enumerate(meta["cameras"]):
    for i in EP: ax[k].plot(np.linspace(0, 1, len(bright[(i, cam)])), bright[(i, cam)], lw=0.7, alpha=0.6)
    ax[k].set(title=f"{cam} brightness", xlabel="normalised time")
plt.tight_layout()

## 5. Automatic checks

In [ ]:
issues = []
for i, r in summary.iterrows():
    if not r.imgs_ok: issues.append((i, "image count != joint count"))
    if r.late: issues.append((i, f"{r.late} frames late (> 1.5x frame period)"))
    if pd.isna(r.grasp_s): issues.append((i, "no grasp detected"))
    elif r.grip_squeeze < 2: issues.append((i, f"weak grip: leader only {r.grip_squeeze} below the follower while holding"))
    if r.n_holds > 1: issues.append((i, f"{r.n_holds} grasps (regrasp)"))
    if r.idle_start_s > 2: issues.append((i, f"{r.idle_start_s}s idle at start"))
    if r.idle_end_s > 3: issues.append((i, f"{r.idle_end_s}s idle at end"))
for i, row in jump.iterrows():
    if row.max() > 10: issues.append((i, f"action jump {row.max()} deg in one frame ({row.idxmax()})"))
for r in img_rows:
    for cam in meta["cameras"]:
        if r[f"{cam}_longest_frozen_run"] > 3: issues.append((r["ep"], f"{cam} camera frozen for {r[f'{cam}_longest_frozen_run']} frames"))
if (summary.grip_end > 15).any() and (summary.grip_end < 15).any():
    issues.append(("all", f"gripper end state inconsistent: open in {list(summary.index[summary.grip_end > 15])}, closed in the rest"))
pd.set_option("display.max_colwidth", None)
pd.DataFrame(issues, columns=["episode", "issue"]) if issues else print("No issues found")